<font size="6" color='grey'> <b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

---

<font size="5" color='grey'> <b>
M04a - Übung A2: Structured Output mit Pydantic
</b></font> </br>

**Lernziel:** Verstehe wie man mit `with_structured_output()` typsichere, validierte Ausgaben vom LLM erhält, statt nur rohe Strings zu parsen.

## Setup & Environment

Umgebung vorbereiten und erforderliche Module laden.

In [ ]:
#@title 🔧 Umgebung einrichten (LOCAL VERSION)
# LOKAL: genai_lib muss bereits installiert sein
# Falls nicht: pip install -e /Users/wagnerg/Development/playground/GenAI_GW/lessons/GenAI/04_modul

import subprocess
import sys

# python-dotenv sicherstellen
try:
    from dotenv import load_dotenv
except ImportError:
    print("📦 Installiere python-dotenv...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv

import os

# API Keys aus .env laden
env_path = '/Users/wagnerg/Development/playground/GenAI_GW/.env'
load_dotenv(env_path)

# Imports
from genai_lib.utilities import check_environment, mprint

print("✅ Umgebung wird vorbereitet...")
print()
check_environment()
print()
print(f"✓ OPENAI_API_KEY gesetzt: {'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY'] != ''}")
print(f"✓ genai_lib importiert erfolgreich")

## Imports

Erforderliche LangChain- und Pydantic-Komponenten importieren.

In [ ]:
# LangChain Importe
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

# Pydantic für strukturierte Datenmodelle
from pydantic import BaseModel, Field

print("✅ Alle Importe erfolgreich")

## Model-Konfiguration

Parameter und Modell-Initialisierung.

In [ ]:
# Parameter
model_provider = "openai"
model_name = "gpt-4o-mini"
temperature = 0.0

# Modell definieren
llm = init_chat_model(model_name, model_provider=model_provider, temperature=temperature)

print(f"✅ Modell initialisiert: {model_name}")
print(f"   Temperature: {temperature}")

# Structured Output: Von Strings zu Objekten

## Das Problem ohne Structured Output

```python
# ❌ Unsicher: Nur String-Parsing
response = llm.invoke("Gib mir Name und Alter als JSON")
# response ist ein String → Fehleranfällig!
```

## Die Lösung: `with_structured_output()`

```python
# ✅ Typsicher: Pydantic-Validierung
class PersonInfo(BaseModel):
    name: str
    alter: int

structured_llm = llm.with_structured_output(PersonInfo)
response = structured_llm.invoke("Gib mir Name und Alter")
# response ist ein PersonInfo-Objekt → Typsicher!
```

## Beispiel 1: Einfaches Pydantic-Modell - Person

In [ ]:
# 1. Pydantic-Modell definieren (wie ein "Formular")
class PersonInfo(BaseModel):
    """Informationen über eine Person"""
    name: str = Field(description="Vollständiger Name der Person")
    alter: int = Field(description="Alter der Person in Jahren")
    beruf: str = Field(description="Aktueller Beruf der Person")

print("✅ PersonInfo-Modell erstellt")
print(f"   Schema: {PersonInfo.model_json_schema()}")

In [ ]:
# 2. Strukturiertes LLM erstellen
structured_llm = llm.with_structured_output(PersonInfo)

print("✅ Strukturiertes LLM erstellt")

In [ ]:
# 3. Anfrage stellen
prompt = "Erstelle eine fiktive Person mit Name, Alter und Beruf"
response = structured_llm.invoke(prompt)

# 4. Ergebnis anschauen
print(f"\n📊 Ergebnis:")
print(f"   Typ: {type(response).__name__}")
print(f"   Name: {response.name}")
print(f"   Alter: {response.alter}")
print(f"   Beruf: {response.beruf}")
print(f"\n✅ Typsicher! Name ist ein {type(response.name).__name__}, Alter ist ein {type(response.alter).__name__}")

## Beispiel 2: Komplexeres Modell - Produkt mit Bewertung

In [ ]:
# Modell für Produktbewertungen
class ProductReview(BaseModel):
    """Bewertung eines Produkts"""
    product_name: str = Field(description="Name des Produkts")
    rating: int = Field(description="Bewertung von 1-5 Sternen", ge=1, le=5)
    pros: list[str] = Field(description="Liste der Vorteile (min. 2)")
    cons: list[str] = Field(description="Liste der Nachteile (min. 1)")
    recommendation: str = Field(description="Empfehlung: 'Ja', 'Nein' oder 'Vielleicht'")

print("✅ ProductReview-Modell erstellt")

In [ ]:
# Strukturiertes LLM für ProductReview
product_llm = llm.with_structured_output(ProductReview)

# Anfrage
prompt = "Bewerte den Laptop 'MacBook Pro' als wärst du ein Technik-Reviewer"
review = product_llm.invoke(prompt)

# Anzeige
print(f"\n📱 Produktbewertung: {review.product_name}")
print(f"   ⭐ Rating: {review.rating}/5")
print(f"\n   ✅ Vorteile:")
for pro in review.pros:
    print(f"      • {pro}")
print(f"\n   ❌ Nachteile:")
for con in review.cons:
    print(f"      • {con}")
print(f"\n   💬 Empfehlung: {review.recommendation}")

## Beispiel 3: Nested Models - Adresse im Modell

In [ ]:
# Verschachteltes Modell: Adresse in Person
class Address(BaseModel):
    """Adressinformationen"""
    street: str = Field(description="Straße und Hausnummer")
    city: str = Field(description="Stadt")
    country: str = Field(description="Land")

class Employee(BaseModel):
    """Mitarbeiter-Information mit Adresse"""
    name: str = Field(description="Vollständiger Name")
    title: str = Field(description="Jobtitel")
    email: str = Field(description="E-Mail-Adresse")
    address: Address = Field(description="Wohnort")

print("✅ Employee-Modell mit verschachtelter Address erstellt")

In [ ]:
# Strukturiertes LLM
employee_llm = llm.with_structured_output(Employee)

# Anfrage
prompt = "Erstelle einen fiktiven Mitarbeiter einer Tech-Firma mit allen Details"
employee = employee_llm.invoke(prompt)

# Anzeige
print(f"\n👤 Mitarbeiter-Information:")
print(f"   Name: {employee.name}")
print(f"   Titel: {employee.title}")
print(f"   Email: {employee.email}")
print(f"\n   📍 Adresse:")
print(f"      {employee.address.street}")
print(f"      {employee.address.city}")
print(f"      {employee.address.country}")

## Beispiel 4: Mit Prompt für strukturierte Ausgabe

In [ ]:
# Modell für Blog-Post Metadaten
class BlogMetadata(BaseModel):
    """Metadaten für einen Blog-Post"""
    title: str = Field(description="Titel des Blog-Posts")
    summary: str = Field(description="Kurze Zusammenfassung (max. 100 Wörter)")
    tags: list[str] = Field(description="Relevante Tags/Keywords")
    category: str = Field(description="Kategorie: Tech, Design, Business oder Andere")
    reading_time_minutes: int = Field(description="Geschätzte Lesedauer in Minuten")

print("✅ BlogMetadata-Modell erstellt")

In [ ]:
# Prompt mit ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist ein Content-Manager. Extrahiere Metadaten aus Blog-Inhalten."),
    ("human", "Analysiere diesen Blog-Post und extrahiere die Metadaten:\n\n{content}")
])

# Chain: Prompt → LLM mit Structured Output
blog_chain = prompt | llm.with_structured_output(BlogMetadata)

print("✅ Blog-Metadaten Chain erstellt")

In [ ]:
# Beispiel Blog-Content
blog_content = """
Wie KI die Welt der Softwareentwicklung verändert

Künstliche Intelligenz revolutioniert die Softwareentwicklung auf vielfältige Weise. 
Entwickler nutzen jetzt KI-gestützte Tools wie GitHub Copilot zum Schreiben von Code,
während Unternehmen maschinelles Lernen für Qualitätssicherung einsetzen.

Diese Entwicklung führt zu schnelleren Release-Zyklen, besserem Code und zufriedeneren Teams.
"""

# Anfrage
metadata = blog_chain.invoke({"content": blog_content})

# Anzeige
print(f"\n📝 Blog-Metadaten:")
print(f"   Titel: {metadata.title}")
print(f"   Kategorie: {metadata.category}")
print(f"   Lesedauer: {metadata.reading_time_minutes} min")
print(f"\n   📌 Tags: {', '.join(metadata.tags)}")
print(f"\n   Zusammenfassung:")
print(f"   {metadata.summary}")

## Vergleich: Verschiedene Ansätze

### 1️⃣ String-Response (unsicher)
```python
response = llm.invoke("Gib Name als JSON")
# Typ: str
# ❌ Keine Validierung, fehleranfällig
```

### 2️⃣ System-Prompt mit JSON (etwas besser)
```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "Antworte im JSON-Format: {\"name\": ..., \"age\": ...}"),
    ("human", "{user_input}")
])
chain = prompt | llm | StrOutputParser()
# Typ: str (noch JSON zum Parsen nötig)
# ⚠️ Bessere Struktur, aber noch nicht typsicher
```

### 3️⃣ `with_structured_output()` (best practice) ✅
```python
class Person(BaseModel):
    name: str
    age: int

structured_llm = llm.with_structured_output(Person)
response = structured_llm.invoke("Gib Name und Alter")
# Typ: Person (Pydantic-Objekt)
# ✅ Typsicher, validiert, mit IDE-Support!
```

## Validation in Aktion

In [ ]:
# Pydantic validiert automatisch!
from pydantic import ValidationError

class StrictPerson(BaseModel):
    name: str
    age: int  # Muss eine Zahl sein!

# ✅ Valider Datensatz
try:
    person1 = StrictPerson(name="Alice", age=30)
    print("✅ Valid: Alice ist 30 Jahre alt")
except ValidationError as e:
    print(f"❌ Invalid: {e}")

# ❌ Ungültiger Datensatz (String statt int)
try:
    person2 = StrictPerson(name="Bob", age="nicht_ein_alter")
    print("✅ Valid: Bob")
except ValidationError as e:
    print(f"❌ Invalid: age muss eine Zahl sein, nicht '{type('nicht_ein_alter').__name__}'")

## 💡 Erkenntnisse

### Warum `with_structured_output()`?

| Aspekt | String-Parser | System-Prompt | Structured Output |
|--------|---------------|---------------|-------------------|
| **Validierung** | ❌ Keine | ⚠️ Optional | ✅ Automatisch |
| **Typsicherheit** | ❌ Nur Text | ❌ Text → JSON | ✅ Python-Objekte |
| **IDE-Support** | ❌ Keine | ❌ Keine | ✅ Autocomplete |
| **Fehlerbehandlung** | ❌ Manuell | ⚠️ Try-Catch nötig | ✅ Pydantic-Errors |
| **Verschachtelung** | ❌ Möglich aber manuell | ⚠️ Komplex | ✅ Native Support |
| **Dokumentation** | ❌ Keine | ⚠️ Im Prompt | ✅ Field descriptions |

### Wann verwenden?

✅ **Verwende Structured Output, wenn du:**
- Strukturierte Daten extrahieren willst (JSON, CSV, etc.)
- Typsicherheit brauchst
- Validierung automatisch erfolgen soll
- Mit Objekten statt Strings arbeiten willst

⚠️ **Nutze String-Parser, wenn du:**
- Fließtext brauchst (Erklärungen, Geschichten)
- Kreative Ausgaben benötigst
- Keine strukturierte Validierung nötig ist

## 📝 Zusammenfassung

✅ **Was wir gelernt haben:**

1. **Pydantic BaseModel** - Definiere deine Datenstruktur
2. **Field & Beschreibungen** - Dokumentiere deine Felder
3. **with_structured_output()** - Verbinde Pydantic mit LangChain
4. **Typsicherheit** - IDE-Support und automatische Validierung
5. **Nested Models** - Komplexe, verschachtelte Strukturen möglich

🚀 **Nächste Schritte:**
- Erstelle eigene Pydantic-Modelle für deine Use-Cases
- Kombiniere mit Chains für End-to-End Workflows
- Nutze Validierung für Fehlerbehandlung
- Experimentiere mit komplexeren Modellen